# 00 · Start here

These notebooks rebuild the study's **model library** from the EDHS 2024-25 Individual
Recode: who is in the cohort, how the three outcomes and every predictor were made, how the
data were split, which predictors were kept, how each model was tuned and checked, and what
each model leans on. Run them in order; each one reads only what the earlier ones wrote, and
ends by checking its results against the study's (in `reference/`).

The study predicts whether a woman reaches **one or more** (ANC1+), **four or more** (ANC4+)
and **eight or more** (ANC8+) antenatal care contacts, from what is known before her care
starts. Three models are the core of the library, **logistic regression, random forest and
XGBoost**, each fitted on **four predictor sets** at **three rungs**.

## Before you start

1. **The data.** Obtain `ETIR8AFL.DTA` (Ethiopia DHS 2024-25, Individual Recode, Stata) from
   The DHS Program and put it in `data/`. It is licensed to you and must not be shared;
   nothing in this folder contains survey rows, and everything the notebooks write stays in
   `outputs/`, `results/` and `quick/`, which are never committed.
2. **The environment.** Python 3.12 or 3.13, then `pip install -r requirements.txt`.
3. **The order.** The table below. The first run of a notebook is the only slow one: every
   model the library trains is saved, and a notebook that is run again skips what is done.

| Notebook | Does | Time (8 cores) |
|---|---|---|
| 01 | the cohort, the outcome ladder and the 21 predictors, from the survey file | 2 min |
| 02 | the candidate pool: every survey column screened, 95 admitted | 3 min |
| 03 | the held-out split and the development folds | under 1 min |
| 04 | a helper model for the selection steps (LightGBM on the 95) | 15 min per rung |
| 05 | near-duplicate predictors, and the clean pool of 86 | 5 min |
| 06 | the four predictor sets: full pool, the 21, the short set, the parsimonious set | 6 to 8 h |
| 07 | logistic regression, tuned on every set and rung | 20 min |
| 08 | random forest, tuned on every set and rung | 2 h |
| 09 | XGBoost, tuned on every set and rung | 1 h |
| 10 | evaluation on the held-out women | 30 min |
| 11 | explanations: what each model leans on | 10 min |
| 12 | the planner's levers and the regions' drivers | 10 min |
| E1-E3 | explainable boosting: tuning, explanations, levers (optional) | 1 h |

Times are rough. Notebook 06 is the long one; it can take the study's predictor sets instead
of re-deriving them (see its first cell).

To run them all without opening Jupyter, from this folder:

```
python scripts/run_notebooks.py            # the core track
python scripts/run_notebooks.py --from 07  # resume after a stop
python scripts/run_notebooks.py --quick    # a smoke run
```

Executed copies, with every table, figure and check, land in `executed/`.

## Two switches, at the top of every notebook

- `QUICK = True` runs the whole chain in about an hour with three search trials per model.
  The numbers will not match the study's; use it to check that everything runs.
- `FOLLOW_STUDY = True` (the default): where a re-derived choice (a selection step, a helper
  model's settings) differs from the study's, the notebook says so and carries the study's
  choice forward, so that the later notebooks reproduce the study's numbers exactly. Set it
  to `False` to carry your own results through instead.

## Folders

```
data/            put ETIR8AFL.DTA here
notebooks/       00-12, the core track
src/anc8/        the analysis package every notebook uses
scripts/         the stage scripts some notebooks run
reference/       the study's results, for the checks
outputs/         what the notebooks write (analysis frames, tables, figures)
results/         the model library: one file per model, set and rung
```

In [ ]:
# Is the environment the one the study used, and is the survey file in place?
import importlib, sys
from pathlib import Path

here = Path.cwd()
project = here.parent if here.name.startswith("notebooks") else here
want = {}
for line in (project / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.startswith("#"):
        name, v = line.split("==")
        want[name.strip()] = v.strip()

MODULE = {"scikit-learn": "sklearn", "python-docx": "docx"}
print(f"python {sys.version.split()[0]}")
for name, v in want.items():
    try:
        m = importlib.import_module(MODULE.get(name, name))
        got = getattr(m, "__version__", "?")
        print(f"  {name:14s} {got:10s} {'' if got == v else f'(the study used {v})'}")
    except ImportError:
        print(f"  {name:14s} MISSING    (pip install -r requirements.txt)")

dta = [p for p in (project / "data").glob("*") if p.name.lower() == "etir8afl.dta"]
print("\nsurvey file:", dta[0] if dta else "NOT FOUND: put ETIR8AFL.DTA in data/")